In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow tensorflowjs matplotlib
import os, tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

tf.get_logger().setLevel('ERROR')

BASE_DIR = '/content/drive/MyDrive'
DATASET_PATH = f'{BASE_DIR}/dataset'
MODEL_PATH   = f'{BASE_DIR}/plant_model.keras'
TFJS_PATH    = f'{BASE_DIR}/plant_model_js'

os.makedirs(TFJS_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

IMG_SIZE, BATCH, EPOCHS_NEW, EPOCHS_FINE = 224, 32, 10, 5

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.2, horizontal_flip=True
)

train_gen = datagen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
                                        batch_size=BATCH, class_mode='categorical',
                                        subset='training', shuffle=True)
val_gen = datagen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
                                      batch_size=BATCH, class_mode='categorical',
                                      subset='validation', shuffle=False)

num_classes = len(train_gen.class_indices)

Found 7674 images belonging to 5 classes.
Found 1917 images belonging to 5 classes.


In [ ]:
def build_model(num_classes):
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(base.input, out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

if os.path.exists(MODEL_PATH):
    model = load_model(MODEL_PATH)
    for layer in model.layers[-20:]:
        if not isinstance(layer, Dropout): layer.trainable = True
    epochs = EPOCHS_FINE
else:
    model = build_model(num_classes)
    epochs = EPOCHS_NEW

callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='val_accuracy', mode='max'),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks)

Epoch 1/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 658s 3s/step - accuracy: 0.9993 - loss: 0.0035 - val_accuracy: 0.9969 - val_loss: 0.0082
Epoch 2/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 586s 2s/step - accuracy: 0.9987 - loss: 0.0036 - val_accuracy: 0.9963 - val_loss: 0.0140
Epoch 3/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 581s 2s/step - accuracy: 0.9983 - loss: 0.0042 - val_accuracy: 0.9990 - val_loss: 0.0040
Epoch 4/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 579s 2s/step - accuracy: 0.9993 - loss: 0.0030 - val_accuracy: 0.9963 - val_loss: 0.0128
Epoch 5/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 668s 3s/step - accuracy: 0.9993 - loss: 0.0035 - val_accuracy: 0.9948 - val_loss: 0.0125


In [ ]:
import tensorflowjs as tfjs

model.save(MODEL_PATH)
!rm -rf {TFJS_PATH}
os.makedirs(TFJS_PATH, exist_ok=True)

tfjs.converters.save_keras_model(model, TFJS_PATH)

with open(os.path.join(TFJS_PATH, 'labels.js'), 'w') as f:
    f.write(f"const CLASS_NAMES = {list(train_gen.class_indices.keys())};")

print(f"✅ MODEL: {MODEL_PATH}")
print(f"✅ TFJS:  {TFJS_PATH}")


failed to lookup keras version from the file,
    this is likely a weight only file
✅ MODEL: /content/drive/MyDrive/plant_model.keras
✅ TFJS:  /content/drive/MyDrive/plant_model_js
